# 1. Phase 2: Data Understanding & Profiling Specification

## 1.1 Objective & Scope
The objective of this phase is to perform a rigorous exploratory data analysis (EDA) and structural audit on the raw transactional log (`UCI Online Retail II`). 

By conducting deep data profiling prior to downstream feature engineering, we validate data integrity, identify domain-specific anomalies, quantify missingness, and assess how raw transaction dynamics align with our **Multi-Tier Churn Definition** established in Phase 1.

## 1.2 Data Understanding Taxonomy
Our analysis is structured around four primary analytical dimensions:

1. **Volume & Schema Audit:** Verifying record counts, data types, memory footprint, and primary/foreign key attributes.
2. **Missingness & Completeness Analysis:** Evaluating missing values (especially unassigned `Customer ID`s) and formulating business-driven imputation or exclusion rules.
3. **Data Integrity & Anomaly Detection:** Identifying negative quantities (returns/cancellation logs), zero/negative unit prices, test/administrative stock codes, and outlier distributions.
4. **Temporal Consistency & Seasonality:** Mapping the chronological coverage, identifying transaction gaps, and validating purchase intervals.

---

## 1.3 Schema Architecture Reference
The raw dataset comprises two operational years of retail transactions with the following schema specifications:

| Feature Name | Logical Data Type | Physical Type | Description & Domain Relevance |
| :--- | :--- | :--- | :--- |
| `Invoice` | Categorical / ID | `str` / `object` | 6-digit transaction identifier. Invoices prefixed with 'C' indicate cancellations. |
| `StockCode` | Categorical / ID | `str` / `object` | Stock keeping unit (SKU) identifier assigned to specific products/services. |
| `Description` | Text | `str` / `object` | Nominal item description. Contains dirty text, administrative noise, and missing values. |
| `Quantity` | Discrete Numeric | `int64` | Transaction line-item quantity per order line. Negative values denote returns/adjustments. |
| `InvoiceDate` | Temporal | `datetime64` | Timestamp of transaction creation (`YYYY-MM-DD HH:MM:SS`). |
| `Price` | Continuous Numeric | `float64` | Unit item price in GBP (£). Zero prices indicate gift items, promotional adjustments, or bad debt. |
| `Customer ID` | Categorical / ID | `float64` / `str` | Unique 5-digit customer identifier. Crucial for customer-level churn aggregation. |
| `Country` | Categorical | `str` / `object` | Geographic location of the customer/order destination. |

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set view options for professional profiling display
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.style.use('seaborn-v0_8-whitegrid')

# Define Data Directory Path
BASE_DIR = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "online_retail_II.csv"

print(f"--> Target Raw Data Path: {RAW_DATA_PATH}")

--> Target Raw Data Path: C:\Users\LOQ\OneDrive\Desktop\PROJECTS\PROGRAMMING\Customer Churn Intelligence & Retention System\data\raw\online_retail_II.csv
